<a href="https://colab.research.google.com/github/Szymon-Jablonski/ml-speech-denoiser/blob/main/denoiser_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Speech denoiser with classic Convolutional Neural Network (CNN) approach

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
!tar -xf /content/drive/MyDrive/Denoiser/CommonVoice27_PL.tar -C /content/
!tar -xf /content/drive/MyDrive/Denoiser/UrbanSound8K.tar -C /content/

In [11]:
!pip install torchaudio soundfile matplotlib

In [12]:
import os
import random
import torch
import torchaudio
import soundfile as sf
import numpy as np
from pathlib import Path

In [13]:

def generate_noisy_speech(
    speech_path,
    noise_path,
    output_dir,
    sample_rate=16000,
    snr_db=5,
    peak_level=0.99,
    filename="sample",
    seed=None
):
    """
    Generate a paired noisy/clean audio example.

    Args:
        speech_path: Path to clean speech WAV/audio file.
        noise_path: Path to environmental noise WAV/audio file.
        output_dir: Directory where output files will be saved.
        sample_rate: Target sample rate (default: 16000 Hz).
        snr_db: Desired signal-to-noise ratio in dB.
        peak_level: Maximum absolute amplitude after normalization.
        filename: Output filename without extension.
        seed: Optional random seed for noise cropping.

    Returns:
        noisy_path, clean_path
    """

    # 1. Load audio
    speech, sr_speech = torchaudio.load(speech_path)
    noise, sr_noise = torchaudio.load(noise_path)

    # 2. Convert stereo/multichannel audio to mono
    speech = speech.mean(dim=0, keepdim=True)
    noise = noise.mean(dim=0, keepdim=True)

    # 3. Resample to the same sample rate
    if sr_speech != sample_rate:
        speech = torchaudio.functional.resample(
            speech, sr_speech, sample_rate
        )

    if sr_noise != sample_rate:
        noise = torchaudio.functional.resample(
            noise, sr_noise, sample_rate
        )

    # 4. Check speech length
    speech_length = speech.shape[1]

    if speech_length == 0:
        raise ValueError("Speech recording is empty.")

    if torch.sqrt(torch.mean(speech ** 2)) < 1e-8:
        raise ValueError("Speech recording is silent.")

    # 5. Make noise exactly the same length as speech
    noise_length = noise.shape[1]

    if noise_length == 0:
        raise ValueError("Noise recording is empty.")

    if noise_length < speech_length:

        # Repeat noise if it is shorter than speech
        repetitions = int(np.ceil(speech_length / noise_length))
        noise = noise.repeat(1, repetitions)

    if noise.shape[1] > speech_length:

        # Select a random segment of the required length
        max_start = noise.shape[1] - speech_length

        rng = random.Random(seed)
        start = rng.randint(0, max_start)

        noise = noise[:, start:start + speech_length]

    # 6. Verify that both signals have identical dimensions
    assert speech.shape == noise.shape, "Audio lengths do not match!"

    # 7. Adjust noise according to the desired SNR
    speech_rms = torch.sqrt(torch.mean(speech ** 2))
    noise_rms = torch.sqrt(torch.mean(noise ** 2))

    if noise_rms < 1e-8:
        raise ValueError("Noise recording is silent.")

    snr_linear = 10 ** (snr_db / 20)

    noise = noise * (speech_rms / (snr_linear * noise_rms))

    # 8. Mix speech and noise
    noisy = speech + noise

    # 9. Normalize both signals using the SAME factor
    peak = torch.max(torch.abs(noisy))

    if peak > 0:
        scale = peak_level / peak
        noisy = noisy * scale
        clean = speech * scale
    else:
        clean = speech.clone()

    # 10. Verify lengths and finite values
    assert noisy.shape == clean.shape
    assert torch.isfinite(noisy).all()
    assert torch.isfinite(clean).all()

    # 11. Save audio files
    os.makedirs(output_dir, exist_ok=True)

    noisy_path = os.path.join(output_dir, f"{filename}_noisy.wav")
    clean_path = os.path.join(output_dir, f"{filename}_clean.wav")

    sf.write(
        noisy_path,
        noisy.squeeze(0).numpy(),
        sample_rate
    )

    sf.write(
        clean_path,
        clean.squeeze(0).numpy(),
        sample_rate
    )

    print(f"Generated: {filename}")
    print(f"Duration: {speech_length / sample_rate:.2f} seconds")
    print(f"Sample rate: {sample_rate} Hz")
    print(f"Number of samples: {speech_length}")
    print(f"Requested SNR: {snr_db} dB")

    return noisy_path, clean_path


In [14]:
noisy_path, clean_path = generate_noisy_speech(
    speech_path="/content/CommonVoice27_PL/pl/clips/common_voice_pl_20554701.mp3",
    noise_path="/content/UrbanSound8K/audio/fold1/7061-6-0-0.wav",
    output_dir="/content/denoiser_dataset/train",
    sample_rate=44100,
    snr_db=5,
    filename="example_001",
    seed=42
)

Generated: example_001
Duration: 7.70 seconds
Sample rate: 44100 Hz
Number of samples: 339747
Requested SNR: 5 dB


In [15]:
import IPython.display as ipd

print("Noisy speech:")
display(ipd.Audio(noisy_path))

print("Clean speech:")
display(ipd.Audio(clean_path))

Noisy speech:


Clean speech:


In [25]:
import pandas as pd

commonvoice_metadata_df = pd.read_csv('/content/CommonVoice27_PL/pl/validated.tsv', sep="\t")
commonvoice_paths = [os.path.join('/content/CommonVoice27_PL/pl/clips/', filename) for filename in commonvoice_metadata_df["path"].dropna()]
commonvoice_paths = [p for p in commonvoice_paths if os.path.isfile(p)]

print(commonvoice_paths[:5])
print(len(commonvoice_paths))

urbansound8k_metadata_df = pd.read_csv('/content/UrbanSound8K/metadata/UrbanSound8K.csv')
urbansound8k_paths = [os.path.join('/content/UrbanSound8K/audio/', f'fold{fold_nr}', filename) for fold_nr, filename in zip(urbansound8k_metadata_df["fold"].dropna(), urbansound8k_metadata_df["slice_file_name"].dropna())]
urbansound8k_paths = [p for p in urbansound8k_paths if os.path.isfile(p)]

print(urbansound8k_paths[:5])
print(len(urbansound8k_paths))


/tmp/ipykernel_1845/1898700188.py:3: DtypeWarning: Columns (4,9) have mixed types. Specify dtype option on import or set low_memory=False.
  commonvoice_metadata_df = pd.read_csv('/content/CommonVoice27_PL/pl/validated.tsv', sep="\t")


['/content/CommonVoice27_PL/pl/clips/common_voice_pl_43211254.mp3', '/content/CommonVoice27_PL/pl/clips/common_voice_pl_24005493.mp3', '/content/CommonVoice27_PL/pl/clips/common_voice_pl_20605240.mp3', '/content/CommonVoice27_PL/pl/clips/common_voice_pl_48446389.mp3', '/content/CommonVoice27_PL/pl/clips/common_voice_pl_20605185.mp3']
138798
['/content/UrbanSound8K/audio/fold5/100032-3-0-0.wav', '/content/UrbanSound8K/audio/fold5/100263-2-0-117.wav', '/content/UrbanSound8K/audio/fold5/100263-2-0-121.wav', '/content/UrbanSound8K/audio/fold5/100263-2-0-126.wav', '/content/UrbanSound8K/audio/fold5/100263-2-0-137.wav']
8732
